# Key Ideas

Below is a summary of the key ideas behind the 2D Lattice Boltzmann implementation for the equation

$$
\frac{\partial \mathbf{v}}{\partial t}
= \eta_{xx}\,\nabla^{2}\mathbf{v}
\;+\;\eta_{xy}\,\bigl(\nabla^{2}\mathbf{v}\times\hat{z}\bigr)
\;+\;\frac{e}{m}\bigl(\mathbf{E}+\mathbf{v}\times\mathbf{B}\bigr).
$$


## 1. D2Q9 Setup

- **Lattice:**  
  A 2D lattice with nine discrete velocity vectors \(\{\mathbf{c}_i\}_{i=0,\dots,8}\).

- **Weights:**  
  Each direction has an associated weight \(w_i\).

- **Speed of Sound:**  
  In the D2Q9 model,
  $$
  c_s^2 = \frac{1}{3}.
  $$


### 2. Macroscopic Variables

- **Density:**
  $$
  \rho = \sum_{i=0}^{8} f_i.
  $$

- **Velocity:**
  $$
  \mathbf{v} = \frac{1}{\rho} \sum_{i=0}^{8} f_i\,\mathbf{c}_i.
  $$


### 3. BGK Collision Operator with Forcing

- **Collision Step:**
  $$
  f_i^* = f_i - \frac{\Delta t}{\tau}\left(f_i - f_i^{\mathrm{eq}}\right) + \Delta t\,F_i(\mathbf{F}),
  $$
  where:
  - \(\tau\) is the relaxation time (related to \(\eta_{xx}\)).
  - \(f_i^{\mathrm{eq}}\) is the equilibrium distribution.
  - \(F_i(\mathbf{F})\) represents the forcing term.


### 4. Cross-Viscous and Electromagnetic Force Terms

- **Forcing Term:**
  $$
  \mathbf{F} = \eta_{xy}\,\Bigl(\nabla^2 \mathbf{v} \times \hat{z}\Bigr) + \frac{e}{m}\Bigl(\mathbf{E} + \mathbf{v}\times \mathbf{B}\Bigr).
  $$

- **Implementation Details:**
  - \(\nabla^2 \mathbf{v}\) is approximated using a finite-difference Laplacian.
  - The cross product \(\nabla^2 \mathbf{v} \times \hat{z}\) is computed by rotating the Laplacian by 90° in the plane.
  - The electromagnetic force is added directly as a body force.


### 5. Streaming Step

- **Streaming:**  
  After collision, the updated distribution \(f_i^*\) is shifted along the discrete velocity vector \(\mathbf{c}_i\):
  $$
  f_i(\mathbf{x} + \mathbf{c}_i \Delta t, t + \Delta t) = f_i^*(\mathbf{x}, t).
  $$

- **Boundary Conditions:**  
  Periodic or other boundary condition

In [3]:
"""
Author: Ashesh Ghosh
Last Update: Mar 19 2025
Comment Update: Updated for Faisal per PRL 118, 226601 (2017)
Description: LBM in 2d with additional forcing
"""

import numpy as np

# ---------------------------------------------------------------------
# 1. Simulation parameters
# ---------------------------------------------------------------------
nx, ny = 128, 128                       # Lattice dimensions
timesteps = 10                          # Number of steps to run
tau = 0.1                               # Relaxation time controlling viscosity ~ eta_xx
eta_xy = 0.05                           # Cross-viscous coefficient
charge_over_mass = 1.0                  # e/m
E_field = np.array([0.0, 0.0])          # External electric field (Ex, Ey)
B_field = np.array([0.0, 0.1])          # Magnetic field out-of-plane => (0, 0.1) in 2D means Bz=0.1
omega = 1.0 / tau                       # BGK collision frequency

# Discrete velocities for D2Q9
c_i = np.array([[ 0, 0],
                [ 1, 0], [ 0, 1], [-1, 0], [ 0,-1],
                [ 1, 1], [-1, 1], [-1,-1], [ 1,-1]], dtype=int)

# Weights
w_i = np.array([4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36])

# Speed of sound squared in D2Q9
cs_sq = 1.0/3.0

# ---------------------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------------------
def equilibrium(rho, vx, vy):
    """
    Compute the D2Q9 equilibrium distribution f_i^eq at each lattice site
    given density rho and velocity (vx, vy).
    """
    feq = np.zeros(9)
    usq = vx*vx + vy*vy
    for i in range(9):
        # ci_dot_u = c_i[i,0]*vx + c_i[i,1]*vy
        ci_dot_u = (c_i[i,0]*vx + c_i[i,1]*vy)
        feq[i] = w_i[i] * rho * (
            1.0 + (ci_dot_u/cs_sq) + 0.5*(ci_dot_u**2/(cs_sq**2)) - 0.5*(usq/cs_sq)
        )
    return feq

def laplacian_velocity(vx, vy):
    """
    Compute the Laplacian of vx, vy using a simple 5-point stencil.
    Returns (lap_vx, lap_vy), each of shape (ny, nx).
    """
    lap_vx = np.zeros_like(vx)
    lap_vy = np.zeros_like(vy)

    # Interior points
    lap_vx[1:-1, 1:-1] = (vx[1:-1, 2:] + vx[1:-1, 0:-2]
                         + vx[2:, 1:-1] + vx[0:-2, 1:-1]
                         - 4.0 * vx[1:-1, 1:-1])
    lap_vy[1:-1, 1:-1] = (vy[1:-1, 2:] + vy[1:-1, 0:-2]
                         + vy[2:, 1:-1] + vy[0:-2, 1:-1]
                         - 4.0 * vy[1:-1, 1:-1])

    # Periodic BCs (simple approach)
    # Left-right boundaries
    lap_vx[:, 0]   = (vx[:, 1] + vx[:, -1] + vx[:, 1] + vx[:, -2] - 4.0 * vx[:, 0])
    lap_vx[:, -1]  = (vx[:, -2] + vx[:, 0] + vx[:, -2] + vx[:, 1] - 4.0 * vx[:, -1])
    lap_vy[:, 0]   = (vy[:, 1] + vy[:, -1] + vy[:, 1] + vy[:, -2] - 4.0 * vy[:, 0])
    lap_vy[:, -1]  = (vy[:, -2] + vy[:, 0] + vy[:, -2] + vy[:, 1] - 4.0 * vy[:, -1])
    
    # Top-bottom boundaries using np.roll for periodicity
    lap_vx[0, :]  = (np.roll(vx[0, :], -1) + np.roll(vx[0, :], 1) + vx[1, :] + vx[-1, :] - 4.0 * vx[0, :])
    lap_vx[-1, :] = (np.roll(vx[-1, :], -1) + np.roll(vx[-1, :], 1) + vx[-2, :] + vx[0, :] - 4.0 * vx[-1, :])
    lap_vy[0, :]  = (np.roll(vy[0, :], -1) + np.roll(vy[0, :], 1) + vy[1, :] + vy[-1, :] - 4.0 * vy[0, :])
    lap_vy[-1, :] = (np.roll(vy[-1, :], -1) + np.roll(vy[-1, :], 1) + vy[-2, :] + vy[0, :] - 4.0 * vy[-1, :])

    return lap_vx, lap_vy

def rotate_90(lapx, lapy):
    """
    Rotate the vector (lapx, lapy) by 90 degrees in-plane:
       (x, y) -> (-y, x).
    """
    return -lapy, lapx

def apply_periodic(index, maxSize):
    """Helper for streaming step (periodic boundaries)."""
    if index < 0:
        return maxSize - 1
    elif index >= maxSize:
        return 0
    else:
        return index

# ---------------------------------------------------------------------
# 3. Allocate and initialize fields
# ---------------------------------------------------------------------
# Distribution function f(i, y, x)
f = np.zeros((9, ny, nx))
# Post-collision distribution
f_star = np.zeros_like(f)

# Initialize density and velocity
rho = np.ones((ny, nx))  # uniform initial density
vx  = np.zeros((ny, nx))
vy  = np.zeros((ny, nx))

# Example: add small perturbation in the center
rho[ny//2, nx//2] = 1.2

# Initialize f to equilibrium
for y in range(ny):
    for x in range(nx):
        feq = equilibrium(rho[y,x], vx[y,x], vy[y,x])
        f[:, y, x] = feq

# ---------------------------------------------------------------------
# 4. Main time-stepping loop
# ---------------------------------------------------------------------
for step in range(timesteps):
    # 4A. Compute macroscopic fields (rho, vx, vy)
    for y in range(ny):
        for x in range(nx):
            local_f = f[:, y, x]
            loc_rho = np.sum(local_f)
            loc_vx  = (local_f[1] - local_f[3] + local_f[5] - local_f[6] - local_f[7] + local_f[8])
            loc_vy  = (local_f[2] - local_f[4] + local_f[5] + local_f[6] - local_f[7] - local_f[8])
            # (Above sums come from c_i, but it's often clearer to do a loop with c_i[i,*].)

            rho[y, x] = loc_rho
            vx[y, x]  = loc_vx / loc_rho
            vy[y, x]  = loc_vy / loc_rho

    # 4B. Compute Laplacian of velocity for cross-viscous term
    lap_vx, lap_vy = laplacian_velocity(vx, vy)
    lap_vx_perp, lap_vy_perp = rotate_90(lap_vx, lap_vy)  # (∇²v) × z-hat

    # 4C. Forcing terms at each cell
    #     F = eta_xy (∇²v × z-hat) + (e/m)( E + v × B ).
    #     Here B is assumed out-of-plane => v × B = (vx, vy, 0)×(0,0,Bz).
    #     That cross product in 2D is ( Bz*vy, -Bz*vx ).
    Fx = np.zeros((ny, nx))
    Fy = np.zeros((ny, nx))

    for y in range(ny):
        for x in range(nx):
            # Cross-viscous part
            Fx[y, x] += eta_xy * lap_vx_perp[y, x]
            Fy[y, x] += eta_xy * lap_vy_perp[y, x]

            # EM part
            # E + v x B (assuming Bz = B_field[1] if we store it as (0,Bz))
            Ex, Ey = E_field
            Bz = B_field[1]
            vxloc = vx[y, x]
            vyloc = vy[y, x]
            # v x B => (vyloc * Bz, -vxloc * Bz)
            Fx[y, x] += charge_over_mass * (Ex + vyloc * Bz)
            Fy[y, x] += charge_over_mass * (Ey - vxloc * Bz)

    # 4D. Collision step (BGK + forcing)
    #     f* = f - omega*(f - f_eq) + F_i
    #     One common forcing approach: F_i ≈ w_i * [ (3/cs^2)(c_i·F) ]  (simple approximation)
    for y in range(ny):
        for x in range(nx):
            loc_rho = rho[y, x]
            loc_vx  = vx[y, x]
            loc_vy  = vy[y, x]
            # Equilibrium
            feq = equilibrium(loc_rho, loc_vx, loc_vy)

            # Simple forcing projection
            fx_local = Fx[y, x]
            fy_local = Fy[y, x]
            # c_i·F
            for i in range(9):
                ci_dot_F = c_i[i,0]*fx_local + c_i[i,1]*fy_local
                # Simple forcing scheme:  F_i = w_i * (3/cs^2) * (c_i·F)
                Fi = w_i[i]*(3.0/cs_sq)*ci_dot_F

                f_star[i, y, x] = (f[i, y, x]
                                   - omega * (f[i, y, x] - feq[i])
                                   + Fi)

    # 4E. Streaming step
    #     f(i, y+cy, x+cx) ← f_star(i, y, x)
    #     with periodic boundaries
    for y in range(ny):
        for x in range(nx):
            for i in range(9):
                cx, cy = c_i[i]
                newx = apply_periodic(x + cx, nx)
                newy = apply_periodic(y + cy, ny)
                f[i, newy, newx] = f_star[i, y, x]

    # (Optional) Output or visualization at intervals
    print(f"Step {step}: max|v| = {np.sqrt(vx**2 + vy**2).max():.4f}")

print("Simulation complete.")

Step 0: max|v| = 0.0000
Step 1: max|v| = 0.0217
Step 2: max|v| = 0.1069
Step 3: max|v| = 14.3216
Step 4: max|v| = 13.7820
Step 5: max|v| = 5.6236
Step 6: max|v| = 369.1511
Step 7: max|v| = 45.4627
Step 8: max|v| = 36.3753
Step 9: max|v| = 20.7599
Simulation complete.
